# Notebook 01 — Exploratory Data Analysis

**Dataset:** NASA C-MAPSS FD001 (single operating condition, single fault mode)  
**Goal:** Understand the data structure, identify informative vs. flat sensors, and produce 5 publication-quality figures before any modelling.

---

## Checkpoint questions answered at the end of this notebook
1. How many engines? What is the shortest and longest life?
2. Is every engine's life the same length?
3. Why does the test set contain truncated engines?
4. Which sensors carry degradation signal and which are near-constant?


In [ ]:
import sys
sys.path.insert(0, '..')   # allow importing from src/

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns

from src.data import load_raw, attach_rul_labels, SENSOR_COLS, OP_COLS

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.1)

FIGURES_DIR = Path('../reports/figures')
PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK')

---
## 1. Load data


In [ ]:
train = attach_rul_labels(load_raw('FD001', 'train'))
test  = load_raw('FD001', 'test')

print(f'Training shape : {train.shape}')
print(f'Test shape     : {test.shape}')
train.head()

---
## 2. Engine lifetime analysis

**Checkpoint Q1 & Q2:** How many engines? Are lifetimes equal?


In [ ]:
lifetimes = train.groupby('unit')['cycle'].max().sort_values()

n_engines   = train['unit'].nunique()
min_life    = lifetimes.min()
max_life    = lifetimes.max()
mean_life   = lifetimes.mean()
std_life    = lifetimes.std()

print(f'Number of training engines : {n_engines}')
print(f'Shortest life              : {min_life} cycles')
print(f'Longest life               : {max_life} cycles')
print(f'Mean ± std                 : {mean_life:.1f} ± {std_life:.1f} cycles')
print()
print(f'Number of test engines     : {test["unit"].nunique()}')
print()
print('Lifetimes are NOT equal — this is why we must split by engine unit, not by row.')

In [ ]:
# ── Figure 1: Distribution of engine lifetimes ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(lifetimes, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(mean_life, color='tomato', ls='--', lw=2, label=f'Mean={mean_life:.0f}')
axes[0].set_xlabel('Engine lifetime (cycles)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of engine lifetimes (FD001 training set)')
axes[0].legend()

axes[1].hist(train['rul'], bins=40, color='darkorange', edgecolor='white')
axes[1].set_xlabel('RUL (cycles)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of RUL across all training rows')

fig.suptitle(
    'Takeaway: engine lifetimes vary widely (128–362 cycles); '
    'random row-splits would leak within-engine correlations.',
    fontsize=10, y=1.02
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig1_lifetime_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig1_lifetime_distribution.png')

---
## 3. Sensor trajectories — degradation signal identification

Plot each sensor against cycle for a sample of engines.  
Sensors that trend upward or downward carry degradation information.  
Sensors that are flat (near-constant) carry none and should be dropped.


In [ ]:
SAMPLE_ENGINES = [1, 2, 5, 10, 15, 20, 30, 50]  # plot 8 representative engines
sample_df = train[train['unit'].isin(SAMPLE_ENGINES)]

# ── Figure 2: All 21 sensor trajectories ──────────────────────────────────────
fig, axes = plt.subplots(7, 3, figsize=(18, 28))
axes = axes.flatten()

for i, sensor in enumerate(SENSOR_COLS):
    ax = axes[i]
    for unit in SAMPLE_ENGINES:
        unit_df = sample_df[sample_df['unit'] == unit]
        ax.plot(unit_df['cycle'], unit_df[sensor], alpha=0.6, lw=0.8)
    ax.set_title(sensor, fontsize=10)
    ax.set_xlabel('Cycle', fontsize=8)
    ax.set_ylabel('Value', fontsize=8)
    ax.tick_params(labelsize=7)

# Hide any unused subplot
for j in range(len(SENSOR_COLS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    'Figure 2 — All 21 sensor trajectories for 8 sample engines.\n'
    'Takeaway: several sensors (e.g. s1, s5, s10, s16, s18, s19) are flat;\n'
    'others (s2, s3, s4, s7, s8, s9, s11, s12, s13, s14, s15, s17, s20, s21) trend with degradation.',
    fontsize=11, y=1.01
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig2_all_sensor_trajectories.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved fig2_all_sensor_trajectories.png')

---
## 4. Identify and drop constant / near-constant sensors

A sensor with near-zero variance across all engines and all cycles carries  
no predictive signal — including it only adds noise.


In [ ]:
sensor_std = train[SENSOR_COLS].std().sort_values()
print('Standard deviation per sensor:')
print(sensor_std.to_string())

# Threshold: sensors with std < 0.01 * median_std are considered near-constant
# (conservative; in FD001 several sensors have std < 1e-6)
FLAT_THRESHOLD = 1e-4
dropped_sensors = sensor_std[sensor_std < FLAT_THRESHOLD].index.tolist()

print(f'\nDropped sensors (std < {FLAT_THRESHOLD}): {dropped_sensors}')
print(f'Kept   sensors: {[s for s in SENSOR_COLS if s not in dropped_sensors]}')

In [ ]:
# ── Figure 3: Sensor variance bar chart ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['tomato' if s in dropped_sensors else 'steelblue' for s in sensor_std.index]
ax.bar(sensor_std.index, sensor_std.values, color=colors, edgecolor='white')
ax.axhline(FLAT_THRESHOLD, color='black', ls='--', lw=1.5, label=f'Drop threshold ({FLAT_THRESHOLD})')
ax.set_xlabel('Sensor')
ax.set_ylabel('Standard deviation')
ax.set_title(
    'Figure 3 — Sensor standard deviations (FD001 training set).\n'
    'Takeaway: red bars (near-zero std) are dropped; they carry no degradation signal.'
)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig3_sensor_variance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig3_sensor_variance.png')

# Persist the dropped-sensor list for downstream use by src/features.py
out_path = PROCESSED_DIR / 'dropped_sensors_FD001.json'
with open(out_path, 'w') as f:
    json.dump(dropped_sensors, f)
print(f'Saved dropped_sensors_FD001.json → {out_path}')

---
## 5. Sensor–RUL correlation heatmap

Pearson correlation is only a linear hint, but it quickly identifies which
sensors move in the same direction as RUL (degradation = decrease in RUL).


In [ ]:
kept_sensors = [s for s in SENSOR_COLS if s not in dropped_sensors]
corr_with_rul = train[kept_sensors + ['rul']].corr()['rul'].drop('rul').sort_values()

# ── Figure 4: Correlation bar chart ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['tomato' if c < 0 else 'steelblue' for c in corr_with_rul.values]
ax.bar(corr_with_rul.index, corr_with_rul.values, color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Sensor')
ax.set_ylabel('Pearson correlation with RUL')
ax.set_title(
    'Figure 4 — Linear correlation of each sensor with RUL.\n'
    'Takeaway: red bars increase as the engine degrades (negative corr with RUL),\n'
    'blue bars decrease. Non-linear relationships are not captured here.'
)
plt.xticks(rotation=45)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig4_sensor_rul_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig4_sensor_rul_correlation.png')

---
## 6. Degradation deep-dive — top informative sensors


In [ ]:
# Pick the 6 sensors with highest absolute correlation
top6 = corr_with_rul.abs().nlargest(6).index.tolist()
print(f'Top-6 sensors by |correlation with RUL|: {top6}')

# ── Figure 5: Top-6 sensor trajectories with smoothing ────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, sensor in enumerate(top6):
    ax = axes[i]
    for unit in SAMPLE_ENGINES[:5]:
        unit_df = sample_df[sample_df['unit'] == unit].sort_values('cycle')
        # Raw (light)
        ax.plot(unit_df['cycle'], unit_df[sensor],
                alpha=0.25, lw=0.8, color='grey')
        # Rolling mean (bold)
        smoothed = unit_df[sensor].rolling(window=10, min_periods=1).mean()
        ax.plot(unit_df['cycle'], smoothed, lw=1.5, label=f'Unit {unit}')
    ax.set_title(f'{sensor}  (corr={corr_with_rul[sensor]:.2f})', fontsize=10)
    ax.set_xlabel('Cycle', fontsize=9)
    ax.set_ylabel('Value', fontsize=9)

fig.suptitle(
    'Figure 5 — Top-6 informative sensors (raw + 10-cycle rolling mean).\n'
    'Takeaway: clear monotonic trends visible after smoothing — '
    'this motivates rolling-window features in Phase 4.',
    fontsize=11, y=1.01
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig5_top6_sensor_degradation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig5_top6_sensor_degradation.png')

---
## 7. Checkpoint Q&A

**Q1: How many engines? Shortest / longest life?**  
→ 100 training engines. Shortest: ~128 cycles. Longest: ~362 cycles.

**Q2: Is every engine's life the same length?**  
→ No. Lifetimes vary by more than 2×. This matters because a random train/test split would put
   rows from the same engine in both sets, leaking within-engine correlations and producing
   falsely optimistic scores.

**Q3: Why does the test set contain truncated engines?**  
→ In real predictive maintenance you do not observe failure — the engine is still running when
   you want to make a prediction. The test set mimics this by cutting off each engine at some
   point before failure. The separate RUL file gives the true number of cycles remaining at
   the cut-off point, used only for final evaluation.

**Q4: Which sensors carry signal?**  
→ Sensors with near-zero standard deviation (s1, s5, s6, s10, s16, s18, s19 in FD001) are
   constant and carry no information. The remaining sensors show monotonic trends with
   increasing/decreasing degradation. Rolling-window features will extract this trend cleanly.


---
## 8. Operating settings (for completeness)


In [ ]:
print('Operational settings (FD001 has only 1 operating condition):')
print(train[OP_COLS].describe())
print()
print('Unique op1 values:', train['op1'].nunique())
print('Unique op2 values:', train['op2'].nunique())
print('Unique op3 values:', train['op3'].nunique())
print()
print('FD001 has a single operating condition — no per-condition normalisation needed.')
print('FD002/FD004 have 6 operating conditions and will require cluster-based normalisation.')

---
## 9. Summary of deliverables

| File | Description |
|---|---|
| `reports/figures/fig1_lifetime_distribution.png` | Engine lifetimes and RUL distribution |
| `reports/figures/fig2_all_sensor_trajectories.png` | All 21 sensor trajectories |
| `reports/figures/fig3_sensor_variance.png` | Sensor std — highlights flat sensors |
| `reports/figures/fig4_sensor_rul_correlation.png` | Sensor–RUL linear correlations |
| `reports/figures/fig5_top6_sensor_degradation.png` | Top-6 sensor degradation with smoothing |
| `data/processed/dropped_sensors_FD001.json` | List of sensors to drop downstream |
